# 3. OpenRouter 에이전트 실행

학습된 Router가 Expert를 선택하고 OpenRouter LLM·validator·patch agent를 실행합니다. 이 노트북만 외부 LLM API를 사용합니다.

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'data' / 'arvo'
ARTIFACT_DIR = ROOT / 'artifacts' / 'phase2e'
UTILITY_PATH = ARTIFACT_DIR / 'router_top2_full5_v4.pkl'
ANCHOR_PATH = ARTIFACT_DIR / 'router_anchor_rare_v2.pkl'
MODEL_PATH = UTILITY_PATH if UTILITY_PATH.exists() else ANCHOR_PATH

from llm_security.config import AppConfig
from llm_security.datasets import load_cases_jsonl
from llm_security.experiment import ExperimentRunner
from llm_security.factory import build_pipeline
from llm_security.models import to_dict
from llm_security.routing import AnchorRareRouter, BudgetedUtilityRouter

## 실행 설정

처음에는 API 호출량을 확인하기 위해 5개 사례만 실행합니다. 전체 test split을 실행하려면 `CASE_LIMIT = None`으로 바꾸세요.

In [ ]:
CASE_LIMIT = 5
config = AppConfig.from_env(ROOT / '.env')
if not MODEL_PATH.exists():
    raise FileNotFoundError('Run 01_train_router.ipynb first.')
router = (
    BudgetedUtilityRouter.load(MODEL_PATH)
    if MODEL_PATH == UTILITY_PATH
    else AnchorRareRouter.load(MODEL_PATH)
)
test_cases = load_cases_jsonl(DATA_DIR / 'cases_test.jsonl')
selected_cases = test_cases if CASE_LIMIT is None else test_cases[:CASE_LIMIT]
print('API key configured:', bool(config.model.api_key))
print('RUN_PAID_EXPERIMENTS:', config.runtime.allow_paid_experiments)
print('Router:', type(router).__name__, MODEL_PATH)
print('Expert models:', {k.value: v for k, v in config.model.expert_models.items()})
print('Validator model:', config.model.validator_model)
print('selected cases:', len(selected_cases), '/', len(test_cases))

## 에이전트 파이프라인 실행

In [ ]:
if not config.runtime.allow_paid_experiments:
    print('Agent run skipped: set RUN_PAID_EXPERIMENTS=1 in .env')
elif not config.model.api_key:
    raise RuntimeError('Set OPENROUTER_API_KEY in .env first.')
else:
    runner = ExperimentRunner(build_pipeline(config, router))
    experiment = runner.run(selected_cases)
    output_path = ROOT / 'experiment-arvo.json'
    runner.save(experiment, output_path)
    pprint(to_dict(experiment.aggregate))
    print('saved experiment:', output_path)